# **9. Расширенные ансамблевые модели (Extended Ensemble Models): агрегирование сильнейших моделей**

* __Цель:__ проверить, улучшает ли VotingRegressor validation-качество относительно сильнейших одиночных моделей предыдущих экспериментов.
* __Задачи:__
  - объединить CatBoost, XGBoost и Random Forest с их лучшими feature sets;
  - сравнить equal-weight voting и individual pipelines по MAE, RMSE, MAPE и R²;
  - подтвердить train-only preprocessing каждого base estimator;
  - сохранить global test split закрытым;
  - обосновать решение не реализовывать рискованный stacking без time-series OOF схемы.
* __Алгоритм выполнения:__
  1. Загрузить результаты эксперимента по расширенным ансамблевым моделям.
  2. Проверить состав и feature sets base estimators.
  3. Сопоставить validation metrics individual models и VotingRegressor.
  4. Проверить alignment validation predictions.
  5. Построить comparison figure.
  6. Выполнить methodological audit и интерпретировать результат.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import Pipeline

from traffic_forecasting.config import (
    EXTENDED_ENSEMBLE_COMPARISON_PATH,
    EXTENDED_ENSEMBLE_FIGURE_PATH,
    EXTENDED_ENSEMBLE_METRICS_PATH,
    EXTENDED_ENSEMBLE_PREDICTIONS_PATH,
    PROCESSED_DATA_PATH,
)
from traffic_forecasting.pipeline import (
    EXTENDED_ENSEMBLE_FEATURE_SETS,
    build_extended_ensemble_candidates,
)
from traffic_forecasting.visualization import plot_extended_ensemble_comparison

EXTENDED_ENSEMBLE_REQUIRED_FILES = (
    PROCESSED_DATA_PATH,
    EXTENDED_ENSEMBLE_METRICS_PATH,
    EXTENDED_ENSEMBLE_COMPARISON_PATH,
    EXTENDED_ENSEMBLE_PREDICTIONS_PATH,
)
missing_files = [path for path in EXTENDED_ENSEMBLE_REQUIRED_FILES if not path.is_file()]
if missing_files:
    missing_text = ", ".join(str(path) for path in missing_files)
    raise FileNotFoundError(
        f"Missing required extended ensemble files: {missing_text}. "
        "Run scripts/run_pipeline.py and "
        "scripts/run_experiments.py --extended-ensembles first."
    )

## **9.1. Архитектура расширенного ансамбля (Extended Ensemble Architecture)**

In [ ]:
candidates = build_extended_ensemble_candidates()
voting_candidate = candidates["voting_regressor"]
architecture_summary = pd.DataFrame(
    [
        {
            "base_estimator": model_name,
            "feature_set": EXTENDED_ENSEMBLE_FEATURE_SETS[model_name],
            "pipeline_type": type(estimator).__name__,
            "has_preprocessing_step": (
                isinstance(estimator, Pipeline) and "preprocessing" in estimator.named_steps
            ),
        }
        for model_name, estimator in voting_candidate.estimators
    ]
)

display(Markdown("### **VotingRegressor base estimator architecture**"))
display(architecture_summary)
assert isinstance(voting_candidate, VotingRegressor)
assert voting_candidate.weights is None
assert architecture_summary["has_preprocessing_step"].all()

## **9.2. Загрузка validation-результатов (Loading Validation Results)**

In [ ]:
metrics = pd.read_csv(EXTENDED_ENSEMBLE_METRICS_PATH)
comparison = pd.read_csv(EXTENDED_ENSEMBLE_COMPARISON_PATH)
predictions = pd.read_csv(
    EXTENDED_ENSEMBLE_PREDICTIONS_PATH,
    parse_dates=["date_time"],
)

output_summary = pd.DataFrame(
    {
        "output": ["metrics", "comparison", "validation_predictions"],
        "rows": [len(metrics), len(comparison), len(predictions)],
        "columns": [metrics.shape[1], comparison.shape[1], predictions.shape[1]],
    }
)

display(Markdown("### **Outputs of the extended ensemble experiment summary**"))
display(output_summary)
display(Markdown("### **Extended ensemble validation metrics**"))
display(metrics)

## **9.3. Сравнение с сильнейшими одиночными моделями (Strongest Individual Model Comparison)**

In [ ]:
voting_result = comparison.query("model == 'voting_regressor'").iloc[0]
best_individual = (
    comparison.query("model_group == 'strongest_individual'").sort_values("rmse").iloc[0]
)

display(Markdown("### **Validation ranking by RMSE**"))
display(
    comparison[
        [
            "validation_rank",
            "model",
            "model_group",
            "feature_set_strategy",
            "mae",
            "rmse",
            "mape",
            "r2",
            "rmse_improvement_vs_best_individual_pct",
        ]
    ]
)

display(Markdown("### **Voting improvement summary**"))
display(
    pd.DataFrame(
        {
            "best_individual_model": [best_individual["model"]],
            "best_individual_rmse": [best_individual["rmse"]],
            "voting_rmse": [voting_result["rmse"]],
            "voting_improvement_pct": [voting_result["rmse_improvement_vs_best_individual_pct"]],
        }
    )
)

## **9.4. Аудит validation predictions (Validation Prediction Audit)**

In [ ]:
prediction_audit = predictions.groupby(["model", "split"], as_index=False).agg(
    rows=("date_time", "size"),
    validation_start=("date_time", "min"),
    validation_end=("date_time", "max"),
    actual_target_sum=("actual_traffic_volume", "sum"),
)

display(Markdown("### **Validation prediction alignment**"))
display(prediction_audit)

assert set(prediction_audit["split"]) == {"validation"}
assert prediction_audit["rows"].nunique() == 1
assert prediction_audit["validation_start"].nunique() == 1
assert prediction_audit["validation_end"].nunique() == 1
assert prediction_audit["actual_target_sum"].nunique() == 1

## **9.5. Визуальное сравнение моделей (Extended Ensemble Comparison Plot)**

In [ ]:
display(Markdown("### **VotingRegressor vs strongest individual models**"))
figure, _ = plot_extended_ensemble_comparison(
    comparison,
    output_path=EXTENDED_ENSEMBLE_FIGURE_PATH,
)
plt.show()

## **9.6. Аудит методологических ограничений (Methodological Constraints Audit)**

In [ ]:
methodological_audit = pd.DataFrame(
    {
        "check": [
            "Metrics contain validation split only",
            "Predictions contain validation split only",
            "Every voting base estimator is a preprocessing pipeline",
            "Voting uses equal weights without validation optimization",
            "Stacking with unsafe default CV is not used",
            "Extended ensemble comparison figure was generated",
        ],
        "passed": [
            set(metrics["split"]) == {"validation"},
            set(predictions["split"]) == {"validation"},
            architecture_summary["has_preprocessing_step"].all(),
            voting_candidate.weights is None,
            "stacking_regressor" not in set(metrics["model"]),
            EXTENDED_ENSEMBLE_FIGURE_PATH.is_file(),
        ],
    }
)

display(Markdown("### **Methodological audit results**"))
display(methodological_audit)

assert methodological_audit["passed"].all()

## **9.7. Анализ и интерпретация результатов (Analysis and Interpretation of Extended Ensemble Results)**

В рамках эксперимента по расширенным ансамблевым моделям была выполнена проверка того, позволяет ли агрегирование предсказаний нескольких сильных моделей повысить качество прогнозирования транспортной нагрузки по сравнению с лучшей одиночной моделью. Данный эксперимент продолжает ранее выполненное сравнение baseline-моделей, core ensemble-моделей, результатов настройки гиперпараметров и экспериментов с различными наборами признаков. Основное внимание уделяется не построению новой независимой модели, а проверке целесообразности объединения уже отобранных сильных решений в единый ансамбль.

**Ключевые результаты:**
1. **Сформирован набор сильнейших индивидуальных моделей для агрегирования.**
   В качестве базовых моделей для расширенного ансамбля были использованы `CatBoostRegressor`, `XGBRegressor` и `RandomForestRegressor`. Эти модели были выбраны на основе результатов предыдущих экспериментов, поскольку они показали наиболее высокое качество среди рассмотренных алгоритмов машинного обучения. Для каждой модели использовался соответствующий набор признаков, показавший наилучший результат в эксперименте по исследованию состава признаков.
2. **Для базовых моделей сохранены индивидуальные стратегии признакового пространства.**
   Модель `CatBoostRegressor` использовала сценарий `temporal_calendar_lag`, модель `XGBRegressor` — сценарий `full`, а модель `RandomForestRegressor` — сценарий `temporal_calendar_lag`. Таким образом, агрегирование выполнялось не над одинаковыми моделями, обученными на одном и том же наборе признаков, а над сильнейшими индивидуальными конфигурациями, найденными ранее.
3. **Агрегирование выполнено с помощью `VotingRegressor`.**
   Для объединения предсказаний был использован `VotingRegressor`, реализующий равновесное усреднение прогнозов базовых моделей. В данном эксперименте веса моделей не подбирались по validation-выборке, что снижает риск переоптимизации под конкретный validation-период.
4. **Сравнение выполнено на единой validation-выборке.**
   Все индивидуальные модели и расширенный ансамбль оценивались на одном и том же validation-периоде. Это обеспечивает корректность сравнения, поскольку различия в метриках обусловлены именно модельной архитектурой и способом агрегирования, а не различием временных интервалов или состава наблюдений. В качестве метрик использовались `MAE`, `RMSE`, `MAPE` и `R²`, при этом основным критерием сравнения выступала `RMSE`.
5. **`VotingRegressor` показал лучший результат по validation RMSE.**
   По результатам эксперимента расширенный ансамбль на основе `VotingRegressor` показал наименьшее значение `RMSE` среди рассмотренных кандидатов. Значение validation `RMSE` для `VotingRegressor` составило около `234.39`, тогда как лучший индивидуальный кандидат, `CatBoostRegressor`, показал validation `RMSE` около `236.28`.
6. **Относительное улучшение оказалось умеренным.**
   Улучшение `RMSE` относительно лучшей индивидуальной модели составило примерно `0.8%`. Это показывает, что расширенный ансамбль действительно смог снизить ошибку прогноза, однако величина выигрыша является ограниченной. Следовательно, `VotingRegressor` можно рассматривать как более точный validation-кандидат, но его преимущество следует сопоставлять с увеличением вычислительной сложности и снижением простоты интерпретации по сравнению с одиночной моделью `CatBoostRegressor`.
7. **Положительный эффект может объясняться различием ошибок базовых моделей.**
   `CatBoostRegressor`, `XGBRegressor` и `RandomForestRegressor` используют разные механизмы построения деревьев и ансамблирования. Поэтому их ошибки на отдельных наблюдениях validation-выборки могут частично компенсировать друг друга. Равновесное усреднение предсказаний снижает влияние отдельных неудачных прогнозов конкретной модели и может повышать устойчивость итогового прогноза транспортной нагрузки.
8. **Preprocessing остался leakage-safe.**
   Каждая базовая модель внутри `VotingRegressor` была представлена в виде отдельного preprocessing pipeline. Это важно, поскольку преобразования признаков должны обучаться только на train-выборке и затем применяться к validation-данным.
9.  **`StackingRegressor` не использовался из-за риска утечки данных.**
    Стандартная схема stacking может использовать неподходящую cross-validation стратегию для временных рядов. В этом случае meta-model может получить информацию, не соответствующую хронологическому порядку прогнозирования. Поэтому в текущем эксперименте stacking не был реализован. Его применение допустимо только при построении time-series-aware out-of-fold predictions, где каждая out-of-fold оценка формируется строго на будущей части относительно обучающего фрагмента без нарушения временного порядка.

**Итоговое методологическое резюме:** эксперимент по расширенным ансамблевым моделям показал, что равновесное агрегирование сильнейших индивидуальных моделей с помощью `VotingRegressor` позволяет получить небольшое улучшение validation `RMSE` относительно лучшей одиночной модели. При этом прирост качества оказался умеренным, поэтому в дальнейшем следует рассматривать два практически значимых варианта: `CatBoostRegressor` как более простую и хорошо воспроизводимую модель и `VotingRegressor` как модель с лучшим validation-результатом.
